# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第7章：F0・フォルマント軌跡のモデル化：GAM/GAMM


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Wieling (2018), *Journal of Phonetics*／Sóskuthy (2017), arXiv:1703.05339。

In [ ]:
# ⏳ 初回は数分かかることがある。特にitsadugは依存パッケージが多い。
install.packages(c("mgcv", "itsadug"), repos = "https://cloud.r-project.org")
library(mgcv)      # 一般化加法モデル（GAM/GAMM）の本体
library(itsadug)   # mgcvの結果を扱いやすくする作図・診断関数群
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ F0軌跡データの生成：

**実際の研究例**：Wieling (2018, *Journal of Phonetics*) は、英語母語話者とオランダ語を母語とする英語学習者（L2話者）の調音の違い（EMAによる舌の動き、100,000点を超える調音データ）を題材にGAMMチュートリアルを公開した。軌跡データを「発話区間を10等分して各時点でt検定を繰り返す」方法で分析すると、(1) 10回検定すれば偶然有意になる確率が跳ね上がる多重比較の問題と、(2) 時点と時点の間の情報（曲線の形そのもの）を捨てているという二重の問題を抱える。GAMMは曲線全体を1つのモデルで推定し、この2つを同時に解決する。

第1-6回とは異なり、ここでは「1発話 = 時間方向に複数点のF0測定」という軌跡データを新たに生成する。Wieling (2018) が用いる正規化時間（0-1）を模す。

In [ ]:
set.seed(1)
n_speaker <- 10
n_trial <- 15
time_points <- seq(0, 1, length.out = 20)  # 発話区間を0(開始)〜1(終了)に正規化し20点で測定

traj_data <- expand_grid(   # 話者×トライアル×時点のすべての組み合わせを作る
  speaker = paste0("S", sprintf("%02d", 1:n_speaker)),
  trial = 1:n_trial,
  time = time_points
) %>%
  mutate(
    # トライアルの前半をrising（上昇調）、後半をfalling（下降調）に割り当てる。
    # expand_grid()はspeaker→trial→timeの順に列を展開するので、この時点で
    # trial列がすでに1..n_trialの値を持っている。それを直接条件分岐に使うのが
    # 一番安全（n_trialが奇数でも自動的に「切り捨て7個/残り8個」のように
    # 綺麗に分かれる。rep(each = n_trial/2)のように自分で個数を数えて
    # 手動でベクトルを組み立てる方式は、n_trialが奇数だと非整数のeachになり、
    # 行数がズレて後続のmutate()がエラーで止まる——実際にこれで一度失敗した）。
    condition = if_else(trial <= n_trial / 2, "rising", "falling"),
    # 話者ごとに一定のF0オフセット（声の高さの個人差）を持たせる。
    # rnorm()に渡すのは行数n()ではなく話者数n_speakerであることに注意。
    # n()を渡しても結果は同じになるが、3000個の乱数を引いて先頭10個しか
    # 使わないという紛らわしい書き方になる（第6章のspeaker_biasと同じ流儀に揃える）。
    speaker_offset = rnorm(n_speaker, 0, 5)[as.numeric(factor(speaker))],
    # 条件ごとに直線的なF0の土台を作り、後で残差にノイズを足す
    base = if_else(condition == "rising",
                    120 + 40 * time,
                    160 - 40 * time)
  )

# ★誤差にAR(1)構造を組み込む。★★これが本節の要点である。
#   単純に rnorm(n(), 0, 3) を毎点ごとに独立に足すと、生成したデータに
#   自己相関が一切無いのに後段でAR(1)補正を実演することになり、
#   「存在しない自己相関をモデルに押し込む」無意味な例になってしまう
#   （実際に本書の旧バージョンはこの誤りを含んでいた）。ここでは
#   1発話（speaker×trialの組）ごとに、時点をまたいで
#   e[t] = rho*e[t-1] + rnorm(1, 0, sd*sqrt(1-rho^2)) という
#   本物のAR(1)過程で残差を生成する。
true_rho <- 0.7
ar_errors <- traj_data %>%
  group_by(speaker, trial) %>%
  group_modify(~ {
    e <- numeric(nrow(.x))
    e[1] <- rnorm(1, 0, 3)
    for (t in 2:length(e)) {
      e[t] <- true_rho * e[t - 1] + rnorm(1, 0, 3 * sqrt(1 - true_rho^2))
    }
    tibble(ar_e = e)
  }) %>%
  ungroup() %>%
  pull(ar_e)
# 分散を sd*sqrt(1-rho^2) にしているのは、AR(1)過程の定常分散を
# 元のノイズ（sd=3）に揃えるためである。こうしないと自己相関を
# 入れるほど全体のばらつきまで変わってしまい、比較がフェアでなくなる。

traj_data$F0 <- traj_data$base + traj_data$speaker_offset + ar_errors

head(traj_data)
nrow(traj_data)  # 10話者 x 15トライアル x 20時点 = 3000行になっているはず

# --- 実行結果の例（実際にRで実行して確認済み）---
# [1] 3000


■ GAMMのフィッティング：

`s(time, by = condition)` で条件ごとに非線形の時間トレンドを推定し、話者差は `s(time, speaker, bs = "fs")`（factor smooth）で吸収する。

In [ ]:
traj_data <- traj_data %>% mutate(condition = factor(condition), speaker = factor(speaker))

model_gamm <- bam(
  # s(time, by=condition, k=10): 条件ごとに独立な非線形カーブを、
  #   最大10個の基底関数（k）で表現する。kが大きいほど曲線が複雑になれるが、
  #   大きすぎると過学習するので gam.check() で妥当性を確認する。
  # s(time, speaker, bs="fs", m=1): factor smooth。話者ごとに似た形だが
  #   微妙にずれたカーブを許容する「ランダム効果のGAM版」に相当する。
  F0 ~ condition + s(time, by = condition, k = 10) +
    s(time, speaker, bs = "fs", m = 1),
  data = traj_data
)

summary(model_gamm)
# edf（実効自由度）が1に近ければほぼ直線、大きいほど複雑な曲線を使っている。

# --- 実行結果（抜粋）---
# Parametric coefficients:
#                  Estimate Std. Error t value Pr(>|t|)
# (Intercept)     140.68066    1.25655 111.958   <2e-16 ***
# conditionrising  -0.08559    0.11092  -0.772     0.44
#
# Approximate significance of smooth terms:
#                            edf Ref.df       F p-value
# s(time):conditionfalling 1.00      1 8951.57  <2e-16 ***
# s(time):conditionrising  1.00      1 8600.50  <2e-16 ***
# s(time,speaker)          29.27     89   52.46  <2e-16 ***
#
# R-sq.(adj) =  0.946   Deviance explained = 94.7%
#
# 【出力の読み方】
# ・条件ごとのスムーズ項（s(time):conditionfalling/rising）はどちらもedfが
#   ちょうど1.00である。第7章7.2節で述べたとおり、これは「非線形性が
#   存在しない証拠」ではなく、「この設定のもとでは罰則付き推定がほぼ
#   直線に縮約した」という意味である。
# ・話者ごとのfactor smooth（s(time,speaker)）はedf=29.27と大きく、
#   話者間で軌跡の形が大きくばらついていることを表す。


■ 概念をつかむ：edfは曲線の複雑さ：

12時点の人工F0データ（真の曲線は緩やかな山形、ノイズSD=6 Hz）に、自由度を固定した回帰スプライン（fx=TRUE）をk=3, 4, 10で当てはめ、edf・残差平方和・新しい値への予測誤差を比べる。edfが大きいほど観測点に近づくが、近づくほど良いとは限らない。

In [ ]:
library(mgcv)
f_true <- function(x) 120 + 40 * sin(pi * x)   # 真の曲線：緩やかな山形（F0, Hz）
x    <- seq(0, 1, length.out = 12)             # 時点12点（正規化時間）
grid <- data.frame(x = seq(0, 1, length.out = 200))
fit_all <- function(d) list(
  "直線（参考）" = gam(y ~ x, data = d),
  "k=3 固定"     = gam(y ~ s(x, k = 3,  fx = TRUE), data = d),  # mgcvはk=2を許さず最小3
  "k=4 固定"     = gam(y ~ s(x, k = 4,  fx = TRUE), data = d),
  "k=10 固定"    = gam(y ~ s(x, k = 10, fx = TRUE), data = d),
  "既定 s(x)"    = gam(y ~ s(x), data = d, method = "REML"))
score <- function(m, y_new)
  c(edf = sum(m$edf) - 1,                      # 切片(1)を除いた曲線部分のedf
    RSS = sum(resid(m)^2),
    RMSE_new   = sqrt(mean((y_new - fitted(m))^2)),
    RMSE_curve = sqrt(mean((predict(m, grid) - f_true(grid$x))^2)))

# (A) 1回分のデータ（seed固定）
set.seed(7)
d <- data.frame(x = x, y = f_true(x) + rnorm(12, sd = 6))   # 観測（ノイズSD = 6 Hz）
y_new <- f_true(x) + rnorm(12, sd = 6)                       # 同じ時点で測り直した値
tabA <- t(sapply(fit_all(d), score, y_new = y_new))
print(round(tabA, 2))

# (B) 同じ手順を500回繰り返した平均（1回分の偶然を除く）
set.seed(2026)
res <- replicate(500, {
  d <- data.frame(x = x, y = f_true(x) + rnorm(12, sd = 6))
  y_new <- f_true(x) + rnorm(12, sd = 6)
  t(sapply(fit_all(d), score, y_new = y_new))
})
print(round(apply(res, c(1, 2), mean), 2))

# --- 実行結果 ---
# (A) 1回分、(B) 500回平均（列: edf, RSS, 新しい値への誤差, 真の曲線との距離）
# edf     RSS RMSE_new RMSE_curve
# 直線（参考） 1.00 1426.81    14.92      12.79
# k=3 固定     2.00  355.20     8.60       5.23
# k=4 固定     3.00  302.67     8.01       5.85
# k=10 固定    9.00   27.11     9.80       6.27
# 既定 s(x)    3.37  266.01     8.58       5.76
#              edf     RSS RMSE_new RMSE_curve
# 直線（参考） 1.0 2693.32    15.29      12.80
# k=3 固定     2.0  316.63     6.63       2.60
# k=4 固定     3.0  283.00     6.84       3.02
# k=10 固定    9.0   72.07     7.99       4.97
# 既定 s(x)    4.4  203.69     6.97       3.31
#
# 【出力の読み方】
# ・edfは直線の1からk=10の9まで増え、残差平方和（RSS）は単調に減る。
#   観測点への当てはまりだけを見ると、複雑な曲線ほど「良い」ことになる。
# ・しかし新しい値への誤差（RMSE_new）と真の曲線との距離（RMSE_curve）は、
#   (B)ではedf=2で最小、edf=9で悪化する。複雑さはノイズへの追従にも使われる。
# ・既定のs(x)はedfをデータに応じて自動調整する（(B)の平均edf=4.40）。
# ・人工データ（真の曲線とノイズを自分で決めた）での結果であり、実データでは
#   gam.check()や交差検証で複雑さの妥当性を確認する。


■ 軌跡の可視化と条件間比較：

In [ ]:
plot_smooth(model_gamm, view = "time", plot_all = "condition", rug = FALSE)
# 2条件のF0軌跡を重ねて描く

plot_diff(model_gamm, view = "time", comp = list(condition = c("rising", "falling")))
# 2条件の「差」を時間軸に沿って描き、信頼区間が0をまたがない区間＝有意な差がある区間を示す


■ 自己相関への対処（AR(1)）：

時系列的に隣接する測定点は誤差も相関しやすい——本節のデータはまさにそのように、AR(1)過程で誤差を生成してある（前のセルを参照）。`bam()`の`rho`引数でAR(1)誤差を組み込む（Wieling, 2018 のチュートリアルに準拠）。

**手順は「rhoを勝手に決めて当てはめる」ではない。** (1) まずAR(1)を考慮しないモデルを当て、(2) その残差の隣接時点間の相関（lag-1自己相関）からrhoを推定し、(3) その推定値でAR(1)モデルを当て直し、(4) 残差の自己相関が実際に小さくなったことを確認する、という順序を踏む。

In [ ]:
# ★itsadugの関数は「素のdata.frame」を前提に書かれており、tidyverseのtibbleを
# そのまま渡すと落ちる。tibbleは df[, "列名"] が1列のtibbleを返すのに対し、
# data.frameはベクトルを返すという違いがあり、start_event()内部の
# order(df[, "列名"], ...) が "cannot xtfrm data frames" というエラーで止まる。
# expand_grid()やmutate()の結果はtibbleなので、ここで明示的に変換しておく。
traj_data <- as.data.frame(traj_data)

traj_data <- start_event(traj_data, column = "time", event = c("speaker", "trial"))
# start_event(): 発話（speaker×trial）ごとの最初の時点にAR誤差の起点フラグを立てる。
# 同時に speaker×trial×time の順に行を並べ替える（AR(1)は行の並び順に沿って
# 誤差の相関を仮定するため、この並べ替えが必要）。

# ---- (1)(2) 無調整モデルの残差からrhoを推定する ----
resid0 <- resid(model_gamm)
not_first <- !traj_data$start.event   # 各発話の最初の時点は「前の時点」を持たない
# 隣接する時点どうしの残差の相関＝lag-1自己相関をrhoの推定値とする
rho_est <- cor(resid0[not_first], resid0[which(not_first) - 1])
cat("残差から推定したrho:", round(rho_est, 3), "\n")

# ---- (3) 推定したrhoでAR(1)モデルを当て直す ----
model_gamm_ar <- bam(
  F0 ~ condition + s(time, by = condition, k = 10) +
    s(time, speaker, bs = "fs", m = 1),
  data = traj_data,
  rho = rho_est,
  AR.start = traj_data$start.event
)

# ---- (4) 自己相関が小さくなったことを確認する ----
# resid()がそのまま返す値はAR補正前の尺度なので、rhoで「白色化」してから
# 自己相関を測る（itsadug::acf_resid()が内部で行っているのと同じ処理）。
raw_resid_ar <- resid(model_gamm_ar)
idx <- which(not_first)
whitened <- raw_resid_ar
whitened[idx] <- raw_resid_ar[idx] - rho_est * raw_resid_ar[idx - 1]
rho_after <- cor(whitened[not_first], whitened[which(not_first) - 1])

cat("調整前モデルの残差のlag-1自己相関:", round(rho_est, 3), "\n")
cat("調整後モデルの残差のlag-1自己相関:", round(rho_after, 3), "\n")
cat("調整前モデルのscale推定値:", round(model_gamm$sig2, 3), "\n")
cat("調整後モデルのscale推定値:", round(model_gamm_ar$sig2, 3), "\n")

summary(model_gamm_ar)

# --- 実行結果 ---
# 残差から推定したrho: 0.68
# 調整前モデルの残差のlag-1自己相関: 0.68
# 調整後モデルの残差のlag-1自己相関: 0.02
# 調整前モデルのscale推定値: 9.187
# 調整後モデルのscale推定値: 9.213
#
# 【出力の読み方】
# ・**無調整モデルの残差には0.68という大きな自己相関が残っている。** 本節の
#   データはAR(1)過程（真のrho=0.7）で誤差を生成してあるので、これは当然の
#   結果である——GAMMは「話者差」「条件差」はモデル化するが、時点間の
#   自己相関までは自動では扱わない。
# ・rho=0.68でAR(1)補正を当て直すと、白色化した残差の自己相関は0.02まで
#   下がる。**これがAR(1)補正が実際に機能していることの直接の証拠**である。
#   自己相関を残したまま「補正した」と言い張っても意味がなく、必ずこの
#   ような確認を挟む。
# ・scale推定値（残差分散に相当）は9.187から9.213とほとんど変わらない。
#   AR(1)補正は「残差の大きさ」を変える処置ではなく、**「残差どうしが
#   独立だという誤った仮定」を取り除く処置**だからである。標準誤差や
#   有意性の判断はこの仮定に依存するため、scaleが変わらなくても
#   補正の意味は大きい。
# ・★rhoは「適当に決めてよい値」ではない。無調整モデルの残差から
#   実測して初めて、そのデータに合った値になる。決め打ちのrhoを使うと、
#   自己相関を補正しすぎたり足りなかったりする——本書の旧バージョンは
#   まさにこの「決め打ちのrho」を使っていた誤りを含んでいた。


■ 二値の時系列：視覚世界パラダイムとGAMM：

視覚世界パラダイムの眼球運動データ（各時点で目標語を見ているか0/1）にも、family = binomialを指定するだけで同じGAMMの枠組みが使える。

In [ ]:
set.seed(70)
n_subj <- 15; n_trial <- 20
time_pts <- seq(0, 1000, by = 50)  # ms

d <- expand_grid(subj = paste0("P", 1:n_subj), trial = 1:n_trial, time = time_pts) %>%
  mutate(
    subj_re = rep(rnorm(n_subj, 0, 0.3), each = n_trial * length(time_pts))[1:n()],
    condition = factor(rep(c("congruent", "incongruent"), length.out = n())),
    subj = factor(subj),
    # 注視確率が時間とともにロジスティック曲線的に立ち上がる（条件で立ち上がりの速さが違う）
    logit_p = -2 + 4 * plogis((time - ifelse(condition == "congruent", 300, 500)) / 100) + subj_re,
    fixation = rbinom(n(), 1, plogis(logit_p))
  )

model_fix <- bam(fixation ~ condition + s(time, by = condition, k = 8) +
                    s(time, subj, bs = "fs", m = 1),
                  data = d, family = binomial)

nd <- expand_grid(time = c(300, 600), condition = c("congruent","incongruent"), subj = "P1")
pred <- predict(model_fix, nd, type = "response", exclude = "s(time,subj)", newdata.guaranteed = TRUE)
cbind(nd[,1:2], pred = round(pred, 3))

# --- 実行結果 ---
# 条件ごとの予測確率（時点300msと600ms）:
#   time   condition  pred
# 1  300   congruent 0.498
# 2  300 incongruent 0.175
# 3  600   congruent 0.860
# 4  600 incongruent 0.718
#
# 【出力の読み方】
# ・擬似データでは、目標語と一致する条件（congruent）のほうが早く注視が
#   立ち上がるように設定してある。300msの時点では一致条件49.8%・不一致条件
#   17.5%と大きく開いているが、600msの時点ではどちらも高水準（86.0%、71.8%）に
#   達し、差が縮まっている。
# ・**曲線の形そのもの（立ち上がりの速さ）に条件差があるという主張は、この
#   時間変化を見て初めて言える。** 単一時点の平均差だけでは「差がいつ生じ、
#   いつ消えるか」が分からない。
# ・試行内の学習・適応も同じ枠組みで扱える。`s(trial, by = condition)`の
#   ように試行番号を軌跡のもう1つの軸として加えればよい。


■ 演習：

1. `k`（基底関数の数）を増減させ、`gam.check()`で妥当性を確認せよ。
2. `plot_diff()`の出力から、2条件の差がどの時間帯で有意かを読み取れ。